# Tapping rhythm simulated export analysis

This notebook reads the committed simulated PsyNet export directly. Simulated tapping validates workflow and analysis code, not human rhythm perception.

In [ ]:
import ast, csv, json, tempfile, zipfile
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 50
attempt = Path.cwd()
export_zip = attempt / "evidence" / "simulated_data.zip"
manifest = json.loads((attempt / "code" / "tapping_rhythm_experiment" / "stimuli" / "manifest.json").read_text())["stimuli"]
with tempfile.TemporaryDirectory() as tmp:
    with zipfile.ZipFile(export_zip) as zf:
        zf.extractall(tmp)
    csv_paths = {p.name: p for p in Path(tmp).rglob("*.csv")}
    participants = pd.read_csv(csv_paths["participant.csv"])
    trials = pd.read_csv(csv_paths["trial.csv"])
def parse_answer(x):
    if not isinstance(x, str) or not x:
        return None
    try:
        return json.loads(x)
    except Exception:
        return ast.literal_eval(x)
answers = [parse_answer(x) for x in trials["answer"].dropna()]
tap_df = pd.DataFrame([a for a in answers if isinstance(a, dict) and a.get("trial_kind") in {"calibration", "main"}])
expected = {"stimulus_id", "trial_kind", "simulated_profile_id", "tap_onsets", "inter_tap_intervals", "n_taps", "calibration_status", "failed", "failure_reason", "dropout"}
missing = expected - set(tap_df.columns)
assert not missing, f"Missing expected fields: {sorted(missing)}"
tap_df[["simulated_profile_id", "trial_kind", "stimulus_id", "n_taps", "calibration_status", "failed", "failure_reason", "dropout"]]


In [ ]:
summary = tap_df.groupby(["simulated_profile_id", "trial_kind"], dropna=False).agg(
    trials=("stimulus_id", "count"),
    mean_taps=("n_taps", "mean"),
    failed_trials=("failed", "sum"),
).reset_index()
summary


In [ ]:
interval_rows = []
for _, row in tap_df.iterrows():
    for interval in row["inter_tap_intervals"]:
        interval_rows.append({"profile": row["simulated_profile_id"], "trial_kind": row["trial_kind"], "interval": interval})
intervals = pd.DataFrame(interval_rows)
intervals.groupby(["profile", "trial_kind"], dropna=False).interval.agg(["count", "mean", "std"]).reset_index()


In [ ]:
ax = tap_df.pivot_table(index="simulated_profile_id", columns="trial_kind", values="n_taps", aggfunc="mean").plot(kind="bar", figsize=(6, 3))
ax.set_ylabel("Mean taps per trial")
ax.set_xlabel("Simulated profile")
ax.set_title("Tap counts by profile and trial kind")
plt.tight_layout()


## Interpretation

The export should show one good profile that passes calibration and reaches main trials, while too-few, off-tempo, noisy, and dropout profiles fail calibration with distinguishable reasons. These are deterministic simulations for workflow and analysis validation only.